In [2]:
import pandas as pd

train_path = "train.csv"
pred_path  = r"kdsh_trackA_data_centric\data\gold\run_id=run_20260112_111511\results_train.csv"

train = pd.read_csv(train_path)
preds = pd.read_csv(pred_path)

# Normalize labels -> y_true
label_map = {
    "consistent": 1,
    "contradict": 0,
    "contradiction": 0,   # in case you used this spelling anywhere
}
train["label_norm"] = (
    train["label"].astype(str).str.strip().str.lower()
)
train["y_true"] = train["label_norm"].map(label_map)

# Normalize predictions -> y_pred
preds["y_pred"] = pd.to_numeric(preds["prediction"], errors="coerce").astype("Int64")

# (Optional) de-dup if needed
train = train.drop_duplicates(subset=["id"])
preds = preds.drop_duplicates(subset=["id"])

# Merge on id
df = train.merge(preds[["id", "y_pred"]], on="id", how="inner")

# Basic sanity checks
missing_pred = set(train["id"]) - set(df["id"])
missing_label = df["y_true"].isna().sum()
if missing_pred:
    print(f"⚠️ Missing predictions for {len(missing_pred)} train IDs (showing up to 10): {sorted(list(missing_pred))[:10]}")
if missing_label:
    print(f"⚠️ {missing_label} rows have unmapped labels. Unique labels:", sorted(train["label_norm"].unique()))

# Drop rows where label or pred is missing
df_eval = df.dropna(subset=["y_true", "y_pred"]).copy()
df_eval["y_true"] = df_eval["y_true"].astype(int)
df_eval["y_pred"] = df_eval["y_pred"].astype(int)

# Accuracy
acc = (df_eval["y_true"] == df_eval["y_pred"]).mean()
print(f"Train accuracy: {acc:.4f}  ({(df_eval['y_true']==df_eval['y_pred']).sum()}/{len(df_eval)})")

# (Optional) confusion counts
tp = ((df_eval.y_true==1) & (df_eval.y_pred==1)).sum()
tn = ((df_eval.y_true==0) & (df_eval.y_pred==0)).sum()
fp = ((df_eval.y_true==0) & (df_eval.y_pred==1)).sum()
fn = ((df_eval.y_true==1) & (df_eval.y_pred==0)).sum()
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")


Train accuracy: 0.6000  (48/80)
TP=29, TN=19, FP=10, FN=22


In [3]:
from sklearn.metrics import classification_report
report = classification_report(df_eval["y_true"], df_eval["y_pred"], target_names=["contradict", "consistent"])

In [5]:
print(report)
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")

              precision    recall  f1-score   support

  contradict       0.46      0.66      0.54        29
  consistent       0.74      0.57      0.64        51

    accuracy                           0.60        80
   macro avg       0.60      0.61      0.59        80
weighted avg       0.64      0.60      0.61        80

TP=29, TN=19, FP=10, FN=22
